## Ontology Query Agent — Server-Side Batch Evaluation (Ground Truth)

This notebook evaluates the **deployed** Ontology Query Agent (VKG path, AgentCore Runtime
`QueryRuntimeArn`) **entirely server-side** with Amazon Bedrock **AgentCore Batch
Evaluations** (`bedrock_agentcore.evaluation.BatchEvaluationRunner`). It mirrors
`2_metadata_query_agent_ondemand_groundtruth_eval.ipynb` (the SemanticRAG metadata query
agent) but targets the **VKG** path: the ontology query agent fetches the ontology from
Neptune, disambiguates query terms against it, generates SQL, runs it on Athena, and maps
the result back to RDF N-Quads.

**No scoring runs locally.** The only client-side work is invoking the agent once per
ground-truth row (unavoidable: the agent must run to produce spans) and reading its
response payload to record cost/latency. All evaluator judging happens in-service.

### Why batch (server-side) instead of the on-demand toolkit
The previous revision of this notebook used the on-demand `Evaluation.run` toolkit (which
filters spans by parsing `cloud.resource_id` and was observed to raise *"No spans found"*)
**plus a local `bedrock_runtime.converse` LLM judge**. The **batch** service discovers
spans by **service name + session id + time range** (not `cloud.resource_id`) and runs the
custom judges **in-service**, so neither local runner is needed.

### Why SESSION-level evaluators
This agent can answer in **one** turn, or ask a **clarifying question** first and answer on
a later turn. A SESSION-level evaluator reads the whole conversation via `{context}` and
scores the **final** answer once — the right granularity for a multi-turn agent. This set is
identical to notebook 2's: the per-turn `Builtin.Correctness` and the TRACE
`QueryAnswerFaithfulness` evaluators were **dropped**, because a TRACE judge errors on a
clarification turn (which makes no model/tool call) and fails the whole session.

### What gets scored (all server-side, all SESSION-level)
**Custom LLM-as-Judge evaluators** (created with `create_evaluator`, scored in-service — no Lambda)

| Evaluator | Level | Placeholders | Checks |
|:--|:--|:--|:--|
| `GoalSuccess` | SESSION | `{context}`, `{assertions}` | The agent achieved the user's goal across the conversation. Custom SESSION judge that replaced the un-editable `Builtin.GoalSuccessRate` (which mis-scored clarification turns on this deterministic-graph agent) |
| `FinalAnswerFaithfulness` | SESSION | `{context}`, `{assertions}` | The conversation's FINAL answer matches the expected answer (threaded into `{assertions}`) |
| `SqlGrounded` | SESSION | `{context}`, `{available_tools}` | Every table/column in the executed SQL maps to a class/property in the retrieved ontology (no hallucinated schema) |

> **How `SqlGrounded` sees the SQL + the ontology without a Lambda.** The deployed VKG agent
> runs a deterministic Tier 2 graph (phase functions), NOT a ReAct tool loop — the ontology
> fetch, slice assembly, and disambiguation are graph phases, and Phase 5 translates SPARQL→SQL
> (Ontop) and runs it on Athena directly. So there are **no** `get_ontology_from_neptune` /
> `disambiguate_query_terms` / `execute_sql_query` model tool calls. The `{context}` placeholder
> is built from the session spans, whose phase outputs carry the retrieved **ontology slice**
> (classes/properties → tables/columns) and the **executed SQL** (Phase 5 output /
> `reasoning.sqlQuery`). The judge reads both from `{context}` — no code-based/Lambda evaluator
> required.

### Prerequisites
- **Notebook 5** (`5_ontology_agent_ac_eval.ipynb`) run to `%store ontology_id_stored`
  (a completed VKG layer the query agent can resolve), or set `EVAL_ONTOLOGY_ID`
- **AgentCore stack** deployed; Ontology Query Agent running
- `bedrock-agentcore>=1.11.0` and `bedrock-agentcore-starter-toolkit==0.3.9`

In [1]:
# Prereq (uncomment if not already installed):
# !pip install bedrock-agentcore-starter-toolkit==0.3.9

## 1. Setup & Credentials

In [2]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Load .env if present, then set defaults
load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

# Create session with AWS profile
session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
print(f"✓ AWS Credentials Verified:")
print(f"  Profile: {os.environ['AWS_PROFILE']}")
print(f"  Account: ...{identity['Account'][-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region: {region}")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


✓ AWS Credentials Verified:
  Profile: huthmac+demo
  Account: ...4087
  User/Role: huthmac-Isengard
  Region: us-east-1


In [3]:
import re

def _redact_account_ids(obj):
    """Recursively mask AWS account IDs in ARN strings to their last 4 digits (********NNNN)."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(\d{8})(\d{4})', r'********\2', obj)
    return obj


def _mask_acct(text):
    """Mask AWS account IDs (12-digit runs) to their last 4 digits — e.g. ********2087 —
    in any string (ARNs, bucket names, plain ids). Never emits the full account number."""
    import re as __re
    if not isinstance(text, str):
        return text
    return __re.sub(r'(\d{8})(\d{4})', r'********\2', text)


In [4]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# Invoke the JWT-inbound runtime with a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    cfg = lam.get_function_configuration(FunctionName=f'{PROJECT_NAME}-mcp-tools')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


## 2. Load the Groundtruth Dataset

Schema (one record per row):

| Field | Type | Description |
|:------|:-----|:------------|
| `Natural_Language_Question` | string | The question fed to the agent |
| `Expected_Answer` | string | The natural-language answer the agent should produce |
| `Expected_SQL_Query` | string | The SQL query that would correctly answer the question |
| `Expected_SQL_Result` | list[dict] | The result rows that SQL should return |

The dataset drives the whole run: one scenario per row is built (section 5), the agent is
invoked once per scenario, and the groundtruth fields (the expected answer + expected
SQL/result) are threaded into each scenario's `assertions` so the server-side SESSION custom
evaluators (`GoalSuccess`, `FinalAnswerFaithfulness`, `SqlGrounded`) can read them.

In [5]:
with open('../data/eval/groundtruth_dataset.json', 'r') as f:
    groundtruth = json.load(f)

# Data-query rows need SQL + expected result; advisory/meta rows (Expected_Tier
# == 'advisory') are answered by the intent-router advisory path and carry only
# a question + Expected_Answer, so they are exempt from the SQL requirement.
BASE_COLS = {'Natural_Language_Question', 'Expected_Answer'}
SQL_COLS = {'Expected_SQL_Query', 'Expected_SQL_Result'}
for i, row in enumerate(groundtruth):
    is_advisory = str(row.get('Expected_Tier', '')).lower() == 'advisory'
    required = BASE_COLS if is_advisory else (BASE_COLS | SQL_COLS)
    missing = required - set(row.keys())
    if missing:
        raise ValueError(f"Row {i} missing columns: {missing}")


MAX_ROWS = int(os.environ.get('MAX_ROWS', '0'))
if MAX_ROWS > 0:
    groundtruth = groundtruth[:MAX_ROWS]
    print(f"⚠ MAX_ROWS={MAX_ROWS} — evaluating a {len(groundtruth)}-row slice (set 0 for all)")

df_gt = pd.DataFrame(groundtruth)

# ── Loud ground-truth quality check ──
# The custom SESSION judges (GoalSuccess, FinalAnswerFaithfulness, SqlGrounded) score against Expected_Answer /
# Expected_SQL_Query / assertions. Rows whose expected fields are still 'PLACEHOLDER' are scored
# against placeholder ground truth — the numbers are only meaningful once these are filled in.
def _is_placeholder(v) -> bool:
    """True when a ground-truth field is empty or the literal 'PLACEHOLDER' sentinel."""
    return v in (None, '', [], 'PLACEHOLDER')

_ph_sql = sum(1 for r in groundtruth if _is_placeholder(r.get('Expected_SQL_Query')))
_ph_ans = sum(1 for r in groundtruth if _is_placeholder(r.get('Expected_Answer')))
print(f"✓ Loaded {len(df_gt)} groundtruth row(s)")
if _ph_sql or _ph_ans:
    print(f"  ⚠ {_ph_sql}/{len(groundtruth)} rows have a PLACEHOLDER Expected_SQL_Query and "
          f"{_ph_ans}/{len(groundtruth)} have a PLACEHOLDER Expected_Answer.")
    print(f"    The custom GoalSuccess / FinalAnswerFaithfulness judges will score "
          f"against placeholders for those rows — fill them in for meaningful scores.")
display(df_gt[['Natural_Language_Question', 'Expected_SQL_Query']])

✓ Loaded 16 groundtruth row(s)
  ⚠ 3/16 rows have a PLACEHOLDER Expected_SQL_Query and 0/16 have a PLACEHOLDER Expected_Answer.
    The custom GoalSuccess / FinalAnswerFaithfulness judges will score against placeholders for those rows — fill them in for meaningful scores.


,Natural_Language_Question,Expected_SQL_Query
0,Show me policies where the insured party is also the policyholder.,"SELECT DISTINCT h.holding_id, h.policy_id, lp.party_id AS insured_party_id\nFROM normalized.holding h\nJOIN normalized.life_participant lp \n ON lp.holding_id = h.holding_id\nJOIN (\n SELECT holding_id, party_id\n FROM normalized.life_participant\n WHERE is_deleted = false\n GROUP BY holding_id, party_id\n HAVING COUNT(DISTINCT participant_sk) > 1\n) dup \n ON dup.holding_id = h.holding_id \n AND dup.party_id = lp.party_id\nWHERE lp.is_deleted = false \n AND h.is_deleted = false"
1,"For each rider, who are the insured participants and what are their roles?","SELECT\n r.rider_id,\n r.rider_name,\n r.rider_code,\n r.rider_type,\n r.rider_status,\n r.holding_id,\n c.coverage_id,\n c.coverage_type AS participant_role,\n c.party_id,\n COALESCE(NULLIF(p.full_name, ''), CONCAT(p.first_name, ' ', p.last_name)) AS participant_name,\n p.party_type,\n c.issue_age,\n c.issue_gender,\n c.underwriting_class\nFROM normalized.rider r\nJOIN normalized.coverage c\n ON r.holding_id = c.holding_id\nLEFT JOIN normalized.party p\n ON c.party_id = p.party_id\nWHERE r.is_deleted = false\n AND c.is_deleted = false\nORDER BY r.rider_id, c.coverage_id\nLIMIT 10"
2,List the top 5 most common party types and their human-readable descriptions.,"SELECT p.party_type, COUNT(*) AS party_count\nFROM normalized.party p\nWHERE p.is_deleted = false\nGROUP BY p.party_type\nORDER BY party_count DESC\nLIMIT 5"
3,"What is the total market value of all active holdings, grouped by party?","SELECT\n p.party_id,\n p.full_name AS party_name,\n COUNT(DISTINCT h.holding_id) AS holding_count,\n SUM(h.market_value) AS total_market_value\nFROM normalized.holding h\nJOIN normalized.coverage c\n ON h.holding_id = c.holding_id\nJOIN normalized.party p\n ON CONCAT('PARTY#', c.party_id) = p.party_id\nWHERE h.holding_status = 'Active'\n AND h.is_deleted = false\n AND c.is_deleted = false\n AND p.is_deleted = false\nGROUP BY p.party_id, p.full_name\nORDER BY total_market_value DESC\nLIMIT 10"
4,"For policies that have a payout schedule, show the policyholder's name, the payout frequency, and the projected next-payout amount.","SELECT\n p.full_name AS policyholder_name,\n COALESCE(pp.premium_mode, c.product_code) AS payout_frequency,\n fa.transaction_amount AS projected_next_payout_amount,\n fa.effective_date AS payout_effective_date,\n fa.activity_type AS payout_type,\n ad.holding_id AS policy_id\nFROM normalized.annuity_detail ad\nJOIN normalized.coverage c\n ON c.holding_id = ad.holding_id\n AND c.is_deleted = false\nJOIN normalized.party p\n ON p.party_id = CONCAT('PARTY#', c.party_id)\n AND p.is_deleted = false\nLEFT JOIN normalized.policy_product pp\n ON pp.product_code = c.product_code\n AND pp.is_deleted = false\nLEFT JOIN normalized.financial_activity fa\n ON fa.holding_id = ad.holding_id\n AND fa.activity_type IN ('Withdrawal', 'Dividend', 'Claim')\n AND fa.is_deleted = false\nWHERE ad.deleted = false\nORDER BY ad.holding_id, fa.effective_date DESC\nLIMIT 10"
5,How many parties are there?,SELECT COUNT(*) AS party_count FROM normalized.party WHERE is_deleted = false
6,List the top 10 coverage products by name.,SELECT product_name FROM normalized.coverage_product WHERE is_deleted = false ORDER BY product_name ASC LIMIT 10
7,"Show the top 10 parties by total holding market value, including the investment product names they hold.","SELECT\n p.party_id,\n p.full_name AS party_name,\n SUM(h.market_value) AS total_market_value,\n ARRAY_JOIN(ARRAY_AGG(DISTINCT\n CASE\n WHEN hs.fundname IS NOT NULL THEN hs.fundname\n WHEN pp.product_name IS NOT NULL THEN pp.product_name\n ELSE 'N/A'\n END\n ), ', ') AS investment_products\nFROM normalized.party p\nJOIN normalized.coverage c\n ON c.party_id = REPLACE(p.party_id, 'PARTY#', '')\n AND c.is_deleted = false\nJOIN normalized.holding h\n ON h.holding_id = c.holding_id\n AND h.is_deleted = false\nLEFT JOIN normalized.holding_suba

## 3. Resolve the AgentCore Ontology Query Runtime

Restore `ontology_id_stored` (stored by notebook 5 — the VKG layer the query agent
resolves) and look up the Ontology Query Runtime ARN (`QueryRuntimeArn`) from the AgentCore
CloudFormation stack outputs.

In [6]:
# ── Resolve the VKG ontology layer to evaluate (best-coverage, validated) ──
# The ground-truth questions span MANY tables (party, policy, holding, coverage,
# rider, ...), so the evaluation ontology must model the full namespace. A
# 1-table smoke-test layer (e.g. from a notebook-5 MAX_TABLES=1 run) cannot answer
# them — the slice judge correctly finds the needed classes missing and the agent
# emits no SQL (0 rows on every row). So:
#   * an explicit EVAL_ONTOLOGY_ID always wins (operator override);
#   * otherwise prefer the completed VKG layer with the MOST dataSources (best
#     ground-truth coverage), NOT merely the newest — a fresh 1-table smoke test
#     must never shadow the full-namespace ontology;
#   * a stored ontology_id_stored (from notebook 5) is honored only if it is the
#     best-coverage layer; otherwise we fall through to it.
_meta_table = session.resource('dynamodb', region_name=region).Table(
    os.environ.get('ONTOLOGY_METADATA_TABLE', f'{PROJECT_NAME}-metadata')
)


def _completed_vkg_layers() -> list:
    """Return completed ``vkg-ontology-*`` items with their dataSource counts.

    Returns:
        List of (layer_id, num_tables, updatedAt) tuples, completed layers only.
    """
    items, scan_kw = [], {
        'FilterExpression': 'contains(id, :p) AND #s = :c',
        'ExpressionAttributeNames': {'#s': 'status'},
        'ExpressionAttributeValues': {':p': 'vkg-ontology', ':c': 'completed'},
    }
    while True:
        page = _meta_table.scan(**scan_kw)
        items.extend(page.get('Items', []))
        if 'LastEvaluatedKey' not in page:
            break
        scan_kw['ExclusiveStartKey'] = page['LastEvaluatedKey']
    out = []
    for it in items:
        n = len(it.get('dataSources') or [])
        upd = it.get('updatedAt') or it.get('createdAt') or ''
        out.append((it['id'], n, upd))
    return out


def _best_coverage_vkg() -> str:
    """Return the completed VKG layer id with the most tables (newest breaks ties)."""
    layers = _completed_vkg_layers()
    if not layers:
        return ''
    layers.sort(key=lambda t: (t[1], t[2]), reverse=True)  # (num_tables, updatedAt)
    return layers[0][0]


def _layer_table_count(layer_id: str) -> int:
    """dataSource count for a specific completed layer id, or -1 if absent/incomplete."""
    if not layer_id:
        return -1
    resp = _meta_table.query(
        KeyConditionExpression='id = :i',
        ExpressionAttributeValues={':i': layer_id},
    )
    for it in resp.get('Items', []):
        if it.get('status') == 'completed':
            return len(it.get('dataSources') or [])
    return -1


# An explicit env override always wins; else the stored id from notebook 5.
_explicit = os.environ.get('EVAL_ONTOLOGY_ID', '')
_stored = ''
try:
    %store -r ontology_id_stored
    _stored = ontology_id_stored
except Exception:
    _stored = ''

_best = _best_coverage_vkg()
if not _best and not _explicit and not _stored:
    raise RuntimeError(
        "No completed VKG (vkg-ontology-*) layer found. Run notebook 5 / the admin UI "
        "first, or set EVAL_ONTOLOGY_ID to a completed VKG layer id."
    )

if _explicit and _layer_table_count(_explicit) >= 0:
    EVAL_ID = _explicit
    print(f"✓ Using EVAL_ONTOLOGY_ID override: {EVAL_ID} "
          f"({_layer_table_count(EVAL_ID)} table(s))")
else:
    # Prefer the best-coverage layer. Honor the stored id only if it IS the
    # best-coverage one (so a 1-table smoke test never shadows the full ontology).
    _best_n = _layer_table_count(_best)
    _stored_n = _layer_table_count(_stored)
    if _stored and _stored_n >= _best_n and _stored_n > 0:
        EVAL_ID = _stored
        print(f"✓ Using stored ontology_id (best coverage): {EVAL_ID} ({_stored_n} table(s))")
    else:
        EVAL_ID = _best
        if _stored and _stored_n < _best_n:
            print(f"⚠ Stored ontology_id {_stored!r} has only {_stored_n} table(s) — "
                  f"using the best-coverage VKG layer instead.")
        print(f"✓ Using best-coverage VKG layer: {EVAL_ID} ({_best_n} table(s))")
    ontology_id_stored = EVAL_ID
    %store ontology_id_stored

if _layer_table_count(EVAL_ID) < 2:
    print(f"  ⚠ WARNING: the chosen ontology models only {_layer_table_count(EVAL_ID)} "
          f"table(s); multi-table ground-truth questions will not be answerable. "
          f"Build a full-namespace ontology (notebook 5 with MAX_TABLES=0).")

# ── Look up Ontology Query Runtime ARN + derive agent_id ──
cfn = session.client('cloudformation', region_name=region)
stack_name = f'{PROJECT_NAME}-agentcore'
try:
    outputs = cfn.describe_stacks(StackName=stack_name)['Stacks'][0]['Outputs']
except Exception:
    stack_name = f'{PROJECT_NAME}-dev-agentcore'
    outputs = cfn.describe_stacks(StackName=stack_name)['Stacks'][0]['Outputs']
output_map = {o['OutputKey']: o['OutputValue'] for o in outputs}

ONTOLOGY_QUERY_RUNTIME_ARN = output_map['QueryRuntimeArn']
# arn:aws:bedrock-agentcore:region:account:runtime/<agent_id>
agent_id = ONTOLOGY_QUERY_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"\n✓ AgentCore Runtime (from stack '{stack_name}'):")
print(f"  Runtime ARN: {_mask_acct(ONTOLOGY_QUERY_RUNTIME_ARN)}")
print(f"  Agent ID:    {agent_id}")

✓ Using EVAL_ONTOLOGY_ID override: vkg-ontology_curated_layer-3d1fde76 (40 table(s))



✓ AgentCore Runtime (from stack 'semantic-layer-dev-agentcore'):
  Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:********4087:runtime/semantic_layer_dev_ontology_query-HKGVpkE6XK
  Agent ID:    semantic_layer_dev_ontology_query-HKGVpkE6XK


## 4. Create Custom (LLM-as-Judge) Evaluators — server-side, no Lambda

All three custom evaluators are **SESSION-level** — each reads the full conversation via
`{context}` across all turns, scoring the final answer once. This is the correct granularity
for a multi-turn agent: a clarify-then-answer conversation is judged on its destination, not
on each intermediate turn. (Same set as notebook 2.)

| Evaluator | Level | Placeholders | Checks |
|:--|:--|:--|:--|
| `GoalSuccess` | SESSION | `{context}`, `{assertions}` | The agent achieved the user's goal across the conversation |
| `FinalAnswerFaithfulness` | SESSION | `{context}`, `{assertions}` | The conversation's FINAL answer matches the expected answer |
| `SqlGrounded` | SESSION | `{context}`, `{available_tools}` | Every table/column in the executed SQL maps to a class/property in the retrieved ontology (no hallucinated schema) |

> **How `FinalAnswerFaithfulness` gets the expected answer.** SESSION evaluators cannot use
> `{expected_response}` (TRACE-only). The expected answer is threaded into `{assertions}` (by
> `eval_multiturn.build_trajectory_assertions`), and the judge extracts it from there.

> **How `SqlGrounded` sees the SQL and the ontology without a Lambda.** The deployed VKG agent
> runs a deterministic Tier 2 graph (phase functions), NOT a tool loop: the ontology fetch /
> slice / disambiguation are graph phases and Phase 5 translates SPARQL→SQL (Ontop) and runs it
> on Athena directly, so there are no `get_ontology_from_neptune` / `disambiguate_query_terms` /
> `execute_sql_query` model tool calls. The `{context}` placeholder is built from the session
> spans, whose phase outputs carry the retrieved **ontology slice** (classes/properties →
> tables/columns) and the **executed SQL** (Phase 5 output / `reasoning.sqlQuery`). The judge
> reads both from `{context}`.

> Re-running this cell creates **new** evaluators (unique name suffix). Delete old ones via
> `delete_evaluator` if you iterate often.

In [7]:
# Custom SESSION LLM-as-Judge evaluators — defined ONCE in
# agents/shared/eval_judges.py and imported here so notebooks 2/6/11 score with
# byte-identical judges (no inline prompt drift). See that module for why the
# judges are SESSION-level and why RAG vs VKG keep separate prompt families.
try:
    from agents.shared.eval_judges import create_custom_judges, JUDGE_MODEL_ID
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_judges import create_custom_judges, JUDGE_MODEL_ID

_cp = session.client('bedrock-agentcore-control', region_name=region)

# family='vkg' selects the VKG / ontology_query prompt set. Returns the ids ORDERED
# [GoalSuccess, FinalAnswerFaithfulness, SqlGrounded]. GoalSuccess is a custom SESSION judge
# used instead of the un-editable Builtin.GoalSuccessRate (on this deterministic-graph agent the
# builtin mistook an intermediate SPARQL/intent span for the answer and mis-scored clarification
# conversations). ToolCallOrdering was removed upstream (no diagnostic signal on a graph agent).
GOAL_SUCCESS_ID, ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID = create_custom_judges(
    control_client=_cp, family='vkg',
)

CUSTOM_EVALUATOR_IDS = [GOAL_SUCCESS_ID, ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID]
print("✓ Custom server-side evaluators created (all SESSION-level, LLM-as-Judge, no Lambda):")
print(f"  GoalSuccess             (SESSION) : {GOAL_SUCCESS_ID}")
print(f"  FinalAnswerFaithfulness (SESSION) : {ANSWER_FAITHFUL_ID}")
print(f"  SqlGrounded             (SESSION) : {SQL_GROUNDED_ID}")


✓ Custom server-side evaluators created (all SESSION-level, LLM-as-Judge, no Lambda):
  GoalSuccess             (SESSION) : GoalSuccess_945bd536-VjsoKqDtWq
  FinalAnswerFaithfulness (SESSION) : FinalAnswerFaithfulness_945bd536-v5PmyP5Iqd
  SqlGrounded             (SESSION) : SqlGrounded_945bd536-PVILMn3oj9


## 5. Build the Dataset (one scenario per row) + agent_invoker

**Per-row scenarios** — each ground-truth question is its own `PredefinedScenario` with its
own session, so we get **per-query** evaluator scores (via `fetch_evaluation_events`) and
clean **per-query** agent token/latency rows.

The `agent_invoker` is the only client-side work: it invokes the synchronous query agent
once per scenario and records the agent's own cost/latency (from `metadata.usage` /
`runtimeMs` and wall-clock) into `AGENT_RUNS`, keyed by the framework session id. Its return
value (`agent_output`) is the answer text the TRACE evaluators score.

In [8]:
# Multi-turn chat-stream invoker + dataset builder.
# The VKG agent's @app.entrypoint dispatches to _chat_stream when the payload carries a
# `turnId` (main.py:1900), reading {message, sessionId, ontologyId, mode}. It reuses ONE
# sessionId across a scenario's turns and keys DDB history off it, so turn N+1 resolves
# turn N's clarification exactly as in production. We parse the AG-UI SSE response and fold
# any clarification option labels into the judge-visible output, recording per-turn
# cost/latency keyed by (session_id, turn_idx).
import time
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput, AgentInvokerOutput)
from bedrock_agentcore.evaluation.runner.dataset_types import Dataset
try:
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)

# Detect the simulation extra once; gate simulated scenarios on it.
try:
    import strands_evals  # noqa: F401
    SIMULATED_ENABLED = True
except ImportError:
    SIMULATED_ENABLED = False
print(f"simulated mode: {'ENABLED' if SIMULATED_ENABLED else 'DISABLED (scripted only)'}")

AGENT_RUNS = {}              # keyed by run_key(session_id, turn_idx)


def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """Drive the VKG agent's CHAT-STREAM path so it reads+persists history itself.

    The SDK reuses one session_id across a scenario's turns; the chat entrypoint keys DDB
    history off the payload sessionId (mode='vkg'), so turn N+1 resolves turn N's
    clarification as in production. We parse the SSE response, fold clarification option
    labels into the judge-visible output, and record per-turn cost/latency keyed by
    (session_id, turn_idx).
    """
    sid = invoker_input.session_id
    # turn index derived from AGENT_RUNS so a notebook re-run (AGENT_RUNS reset above) is safe
    turn_idx = sum(1 for k in AGENT_RUNS if k.startswith(f"{sid}#turn"))
    message = (invoker_input.payload if isinstance(invoker_input.payload, str)
               else json.dumps(invoker_input.payload))
    payload = build_chat_payload(message=message, session_id=sid,
                                 ontology_id=EVAL_ID, turn_idx=turn_idx)
    payload["mode"] = "vkg"  # VKG path (build_chat_payload defaults to semantic-rag)
    start = time.time(); err = None; parsed = {}
    try:
        raw = _invoke_runtime(ONTOLOGY_QUERY_RUNTIME_ARN, sid,
                              json.dumps(payload).encode("utf-8"))
        parsed = parse_chat_stream_sse(raw.decode("utf-8", errors="replace"))
        err = parsed.get("error")
    except Exception as exc:  # noqa: BLE001
        err = str(exc); print(f"  ⚠ [{sid} t{turn_idx}] {err}")
    _rows = parsed.get("rows", []) or []
    AGENT_RUNS[run_key(sid, turn_idx)] = {
        "scenario_session": sid, "turn_idx": turn_idx, "message": message,
        "answer": parsed.get("answer", ""), "agent_sql": parsed.get("sql", ""),
        # Executed-vs-generated SQL + degrade reason + the actual result rows, so
        # the result file is self-contained evidence (NL question + SQL + result).
        "executed_sql": parsed.get("executed_sql", ""),
        "degraded": parsed.get("degraded"),
        "clarified": bool(parsed.get("clarification")),
        "rows": _rows, "result_row_count": len(_rows), "result_sample": _rows[:3],
        "usage": parsed.get("usage", {}),
        "runtime_ms": parsed.get("runtime_ms"),
        "wall_clock_s": round(time.time() - start, 2), "invoke_error": err,
    }
    state = "clarify" if parsed.get("clarification") else ("answer" if parsed.get("sql") else "none")
    print(f"  {'OK' if err is None else 'XX'} [{sid} t{turn_idx}] "
          f"{AGENT_RUNS[run_key(sid, turn_idx)]['wall_clock_s']}s . {state}")
    return AgentInvokerOutput(agent_output=format_agent_output(parsed) or (err or ""))

_specs = [parse_multiturn_row(r, index=i) for i, r in enumerate(groundtruth)]
dataset = Dataset(scenarios=build_scenarios(
    _specs, ontology_id=EVAL_ID, simulated_enabled=SIMULATED_ENABLED))
EXPECTED_TRAJECTORY = []  # deterministic graph; Phase 5 translates SPARQL→SQL (no model tools)
print(f"✓ Dataset: {len(dataset.scenarios)} scenario(s) "
      f"({sum(1 for s in _specs if len(s['turns'])>1)} multi-turn)")

simulated mode: ENABLED
✓ Dataset: 16 scenario(s) (2 multi-turn)


## 6. Run the Batch Evaluation (server-side)

`run_dataset_evaluation` invokes the agent for each scenario (via `agent_invoker`), waits
`ingestion_delay_seconds` for CloudWatch span ingestion, submits a single
`StartBatchEvaluation` job over the **SESSION-only custom** evaluators, and polls to
completion. All scoring happens in-service. Span discovery is by **service name + session id
+ time range**.

In [9]:
# k-run batch runner: run the WHOLE batch EVAL_K times and AVERAGE
# the per-evaluator aggregate scores in cell 17, so the headline VKG numbers are robust to
# LLM-judge + agent stochasticity. Each run clears + repopulates AGENT_RUNS via agent_invoker
# and snapshots that run's aggregate scores, per-query events, agent_runs, and cost/latency.
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig, BatchEvaluatorConfig, CloudWatchDataSourceConfig,
)
try:
    from agents.shared.eval_multiturn import group_runs_by_session
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import group_runs_by_session

SERVICE_NAME = f"{agent_id}.DEFAULT"
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"
print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"LOG GROUPS   : {SPANS_LOG_GROUP}, {RUNTIME_LOG_GROUP}")

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
    # Multi-turn scenarios emit more spans, and clarification turns ingest a beat
    # later; 420s (matching nb2) gives short clarify turns time to ingest their
    # answer span before scoring. 180s risked dropping late multi-turn spans.
    ingestion_delay_seconds=420,
)

# SESSION-only evaluator set (matches notebook 2). Builtin.Correctness and the TRACE
# QueryAnswerFaithfulness are intentionally DROPPED: TRACE evaluators error on a
# clarification turn (no model/tool span) and fail the whole session. The 3 custom SESSION
# judges (GoalSuccess, FinalAnswerFaithfulness, SqlGrounded) read the full multi-turn
# {context}/{assertions}. GoalSuccess is used instead of the un-editable Builtin.GoalSuccessRate.
ALL_EVALUATOR_IDS = [
    *CUSTOM_EVALUATOR_IDS,            # SESSION — GoalSuccess, FinalAnswerFaithfulness, SqlGrounded
]

# Map evaluator id → display name for the per-run aggregate scores.
_EVAL_NAME = {GOAL_SUCCESS_ID: 'GoalSuccess',
              ANSWER_FAITHFUL_ID: 'FinalAnswerFaithfulness',
              SQL_GROUNDED_ID: 'SqlGrounded'}
# AgentCore eval EVENTS carry gen_ai.evaluation.name = the evaluator NAME (the id
# minus its trailing '-<hash>' segment), NOT the full evaluatorId. Alias those names
# to the clean display name so per-query rows + per_scenario_goal_mean resolve
# correctly (else they fall through to the hashed string and the 'GoalSuccess' filter
# misses, leaving per_scenario_goal_mean empty).
_EVAL_NAME.update({_eid.rsplit('-', 1)[0]: _disp for _eid, _disp in list(_EVAL_NAME.items())})

# k-run repetition knob (read-only, from notebooks/.env). EVAL_K=1 → single run (old behaviour).
EVAL_K = int(os.environ.get('EVAL_K', '3'))
batch_runner = BatchEvaluationRunner(region=region)


def _scenario_id_for_session(sess):
    """Map a framework session id (scenario_id + '-' + uuid) back to its scenario_id."""
    for s in _specs:
        if sess and sess.startswith(s['scenario_id'] + '-'):
            return s['scenario_id']
    return sess or ''


def _aggregate_scores(result) -> dict:
    """Pull the per-evaluator average scores out of a finished batch result.

    Parameters:
        result: the BatchEvaluationRunner result for one run.
    Returns:
        {display_name: average_score|None} over ALL_EVALUATOR_IDS.
    """
    scores = {}
    ev = result.evaluation_results
    if ev is not None:
        for es in (ev.evaluator_summaries or []):
            st = es.statistics
            name = _EVAL_NAME.get(es.evaluator_id, es.evaluator_id)
            scores[name] = (round(st.average_score, 4)
                            if st and st.average_score is not None else None)
    return scores


def _agent_perf_snapshot() -> dict:
    """Summarise the CURRENT AGENT_RUNS into one run's cost/latency totals."""
    lat = [r['wall_clock_s'] for r in AGENT_RUNS.values() if r.get('wall_clock_s') is not None]

    def _usum(key: str) -> int:
        return sum(int((r.get('usage') or {}).get(key) or 0) for r in AGENT_RUNS.values())

    return {'turns': len(AGENT_RUNS),
            'avg_wall_clock_s': round(sum(lat) / len(lat), 2) if lat else None,
            'input_tokens': _usum('inputTokens'), 'output_tokens': _usum('outputTokens'),
            'total_tokens': _usum('totalTokens')}


def _fetch_events(result) -> list:
    """Fetch this run's per-query evaluator events (output stream lags COMPLETED — retry)."""
    for _ in range(6):
        try:
            evs = batch_runner.fetch_evaluation_events(result)
            if evs:
                return evs
        except (LookupError, ValueError):
            pass
        time.sleep(20)
    return []


def _event_rows_for_run(result) -> list:
    """Flatten this run's evaluator events into per-(scenario, evaluator) score rows."""
    grouped = group_runs_by_session(AGENT_RUNS)

    def _f(e, key):
        return e[key] if key in e else (e.get('attributes', {}) or {}).get(key)

    rows = []
    for e in _fetch_events(result):
        sess = _f(e, 'session.id') or _f(e, 'gen_ai.session.id')
        turns = grouped.get(sess, [])
        name = _f(e, 'gen_ai.evaluation.name')
        rows.append({
            'scenario_id': _scenario_id_for_session(sess),
            'question': (turns[0]['message'] if turns else ''),
            'evaluator': _EVAL_NAME.get(name, name),
            'score': _f(e, 'gen_ai.evaluation.score.value'),
            'label': _f(e, 'gen_ai.evaluation.score.label'),
        })
    return rows


def _run_one_batch(run_idx: int) -> dict:
    """Run the SESSION-only batch once (fresh AGENT_RUNS); return a per-run snapshot dict."""
    AGENT_RUNS.clear()  # agent_invoker repopulates this per turn for THIS run
    cfg = BatchEvaluationRunConfig(
        batch_evaluation_name=f"ontquery_gt_batch_r{run_idx}_{uuid.uuid4().hex[:8]}",
        description=f"Server-side SESSION-only ground-truth eval of the ontology query agent, "
                    f"run {run_idx}/{EVAL_K}.",
        evaluator_config=BatchEvaluatorConfig(evaluator_ids=ALL_EVALUATOR_IDS),
        data_source=batch_data_source,
        max_concurrent_scenarios=3,   # synchronous agent, up to 900s/row
        # VKG rows are multi-phase (SPARQL→Ontop→Athena) and slow; k=3 × 16 rows can
        # exceed 3600s. 5400s keeps a full k-run from being marked incomplete.
        polling_timeout_seconds=5400,
        polling_interval_seconds=30,
    )
    print(f"\n=== Run {run_idx}/{EVAL_K}: {cfg.batch_evaluation_name} ===")
    res = batch_runner.run_dataset_evaluation(
        config=cfg, dataset=dataset, agent_invoker=agent_invoker)
    print(f"  ✓ status: {res.status}")
    if res.agent_invocation_failures:
        print(f"  ⚠ Agent invocation failures: {len(res.agent_invocation_failures)}")
    snap = {'run': run_idx, 'batch_id': res.batch_evaluation_id,
            'batch_arn': res.batch_evaluation_arn, 'status': str(res.status),
            'scores': _aggregate_scores(res), 'agent_perf': _agent_perf_snapshot(),
            'events': _event_rows_for_run(res),
            'agent_runs': list(AGENT_RUNS.values())}
    snap['_result'] = res  # kept in-memory only (not serialised) for the last-run detail cell
    print(f"  run {run_idx} scores: {snap['scores']}")
    return snap


# ── Resumable k-run loop ─────────────────────────────────────────────────────
# Each completed run's snapshot is persisted to a resume file the MOMENT it
# finishes, so an interrupted session (kernel / env teardown) loses at most the
# in-flight run, not the whole k=3. Re-executing this cell reloads finished runs
# and only runs the missing indices. The resume file is keyed by (EVAL_ID, EVAL_K)
# so a different layer or k starts clean. Delete it to force a fresh run.
_resume_file = f"../data/eval/results/.resume_ontology_query_kmean_{EVAL_ID}_k{EVAL_K}.json"


def _load_resume() -> list:
    """Return previously-completed run snapshots (without the non-serialisable _result)."""
    if not os.path.exists(_resume_file):
        return []
    try:
        with open(_resume_file) as f:
            data = json.load(f)
        if data.get('eval_id') == EVAL_ID and data.get('eval_k') == EVAL_K:
            return data.get('runs', [])
    except Exception:  # noqa: BLE001 — a corrupt resume file must not block a fresh run
        pass
    return []


def _save_resume(runs: list) -> None:
    """Atomically persist completed run snapshots (strip the live _result object)."""
    serialisable = [{k: v for k, v in r.items() if k != '_result'} for r in runs]
    tmp = _resume_file + '.tmp'
    with open(tmp, 'w') as f:
        json.dump({'eval_id': EVAL_ID, 'eval_k': EVAL_K, 'runs': serialisable},
                  f, indent=2, default=str)
    os.replace(tmp, _resume_file)  # atomic rename — never leaves a half-written file


print(f"\nEvaluators : {ALL_EVALUATOR_IDS}")
print(f"Scenarios  : {len(dataset.scenarios)}   ·   EVAL_K = {EVAL_K}")

RUNS = _load_resume()           # per-run snapshots; resumed ones lack the live _result
_done = {r['run'] for r in RUNS}
if _done:
    print(f"↻ Resuming — {len(_done)} run(s) already complete: {sorted(_done)} "
          f"(delete {os.path.basename(_resume_file)} to force a clean run)")
print("Starting k-run batch evaluation (invokes the agent per scenario, then evaluates server-side)...")

for _i in range(1, EVAL_K + 1):
    if _i in _done:
        print(f"  ✓ run {_i}/{EVAL_K} already complete (resume file) — skipping")
        continue
    RUNS.append(_run_one_batch(_i))
    _save_resume(RUNS)          # persist immediately so a kill after this run keeps it
    print(f"  ↳ run {_i} persisted to {os.path.basename(_resume_file)}")

RUNS.sort(key=lambda r: r['run'])
# The last-run detail cell needs a live batch result; resumed runs don't carry one
# (it isn't serialisable). batch_result is None when the final run came from disk —
# cell 17 guards on this and still writes the k-mean headline from RUNS.
batch_result = RUNS[-1].get('_result') if RUNS else None

print(f"\n✓ Completed {EVAL_K} run(s)"
      + ("" if batch_result is not None else " (last run resumed from disk — "
         "per-query last-run detail will be skipped; k-mean summary is unaffected)"))


SERVICE_NAME : semantic_layer_dev_ontology_query-HKGVpkE6XK.DEFAULT
LOG GROUPS   : aws/spans, /aws/bedrock-agentcore/runtimes/semantic_layer_dev_ontology_query-HKGVpkE6XK-DEFAULT



Evaluators : ['GoalSuccess_945bd536-VjsoKqDtWq', 'FinalAnswerFaithfulness_945bd536-v5PmyP5Iqd', 'SqlGrounded_945bd536-PVILMn3oj9']
Scenarios  : 16   ·   EVAL_K = 3
↻ Resuming — 1 run(s) already complete: [1] (delete .resume_ontology_query_kmean_vkg-ontology_curated_layer-3d1fde76_k3.json to force a clean run)
Starting k-run batch evaluation (invokes the agent per scenario, then evaluates server-side)...
  ✓ run 1/3 already complete (resume file) — skipping

=== Run 2/3: ontquery_gt_batch_r2_0b5beb39 ===


  OK [gt-row-02-60955400-dca2-4fec-a60c-b6f172182501 t0] 39.72s . answer


  OK [gt-row-00-a6b510f1-be66-44a8-b07e-169a433e6515 t0] 43.54s . answer


  OK [gt-row-01-e4dfc69b-664c-4df4-95c6-3be034f5dcd9 t0] 89.82s . answer


...

  OK [mt-stable-options-73455c7e-0ad0-44ca-aa48-7ec597ea7142 t2] 20.21s . answer


  OK [gt-row-14-2eecae09-aea8-4ce0-b5a6-eefbedaceb9c t0] 21.84s . none


  OK [gt-row-15-de2b6dbc-154d-4cf2-b215-f81470b8893e t0] 27.06s . none


  ✓ status: COMPLETED


  run 3 scores: {'SqlGrounded': 1.0, 'GoalSuccess': 0.69, 'FinalAnswerFaithfulness': 0.69}
  ↳ run 3 persisted to .resume_ontology_query_kmean_vkg-ontology_curated_layer-3d1fde76_k3.json

✓ Completed 3 run(s)


## 7. Results — aggregate scores, per-query detail, agent cost/latency

Three layers:
1. **Aggregate per-evaluator scores** — from `batch_result.evaluation_results` (in-service).
2. **Per-query evaluator scores** — `fetch_evaluation_events()` reads one event per
   turn-per-evaluator (score, label, explanation) from the batch output log stream, joined
   back to each question by session id.
3. **Per-query agent cost/latency** — from `AGENT_RUNS` (the agent's own tokens / runtimeMs /
   wall-clock; the batch result never carries these).

In [10]:
# k-run mean scores + last-run per-query detail + preserved JSON dumps.
os.makedirs('../data/eval/results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# ════════════════════════════════════════════════════════════════════════════
# A. k-run MEAN aggregate scores (the headline) — averaged over RUNS from cell 15.
# ════════════════════════════════════════════════════════════════════════════
import statistics as _stats

_EVAL_ORDER = ['GoalSuccess', 'FinalAnswerFaithfulness',
               'SqlGrounded']


def _mean_std(values: list):
    """Return (mean, std, n) over the non-None numeric values (std=0.0 when n<2)."""
    nums = [v for v in values if isinstance(v, (int, float))]
    if not nums:
        return None, None, 0
    mean = round(sum(nums) / len(nums), 4)
    std = round(_stats.pstdev(nums), 4) if len(nums) > 1 else 0.0
    return mean, std, len(nums)


mean_rows = []
for ev_name in _EVAL_ORDER:
    per_run = [r['scores'].get(ev_name) for r in RUNS]
    mean, std, n = _mean_std(per_run)
    mean_rows.append({'Evaluator': ev_name, 'MeanScore': mean, 'Std': std,
                      'Runs': n, 'PerRun': [r['scores'].get(ev_name) for r in RUNS]})
df_mean = pd.DataFrame(mean_rows)


def _perf_mean(key: str):
    return _mean_std([r['agent_perf'].get(key) for r in RUNS])[0]

agent_perf_mean = {
    'avg_wall_clock_s': _perf_mean('avg_wall_clock_s'),
    'input_tokens': _perf_mean('input_tokens'),
    'output_tokens': _perf_mean('output_tokens'),
    'total_tokens': _perf_mean('total_tokens'),
}

# Per-scenario mean GoalSuccess across runs.
_per_scn = {}
for r in RUNS:
    for e in r['events']:
        if e['evaluator'] == 'GoalSuccess' and isinstance(e['score'], (int, float)):
            _per_scn.setdefault(e['scenario_id'], []).append(e['score'])
per_scenario_goal_mean = {sid: round(sum(v) / len(v), 4) for sid, v in _per_scn.items()}

# Per-scenario agent detail across all k runs (emitted SQL / mean tokens / mean latency).
# Downstream notebook 9 reads this to build its apples-to-apples matched subset (rows where
# BOTH arms emitted SQL) for this arm WITHOUT re-running this notebook's agent. We key off the
# per-run agent_runs snapshots captured in cell 15, folding multi-turn sessions to their FINAL
# turn (the answer-bearing turn) and averaging tokens/latency over the runs that produced them.
def _scn_id_from_session(sess):
    for s in _specs:
        if sess and sess.startswith(s['scenario_id'] + '-'):
            return s['scenario_id']
    return sess or ''
_psa = {}
for r in RUNS:
    by_sess = {}
    for rec in r['agent_runs']:
        by_sess.setdefault(rec['scenario_session'], []).append(rec)
    for sess, recs in by_sess.items():
        recs.sort(key=lambda x: x.get('turn_idx', 0))
        final = recs[-1]
        sid = _scn_id_from_session(sess)
        d = _psa.setdefault(sid, {'sql_runs': 0, 'tokens': [], 'latency': [],
                                  'gen_sql': '', 'exec_sql': '', 'rows': None,
                                  'sample': [], 'degraded': None})
        if final.get('agent_sql'):
            d['sql_runs'] += 1
        d['tokens'].append(int((final.get('usage') or {}).get('totalTokens') or 0))
        d['latency'].append(sum(x.get('wall_clock_s') or 0 for x in recs))
        # Keep one representative SQL + result for the evidence record: prefer a
        # run whose final turn actually executed SQL over a degraded/no-SQL one.
        if final.get('agent_sql') and (not d['gen_sql'] or final.get('executed_sql')):
            d['gen_sql'] = final.get('agent_sql', '')
            d['exec_sql'] = final.get('executed_sql', '')
            d['rows'] = final.get('result_row_count')
            d['sample'] = final.get('result_sample', [])
            d['degraded'] = final.get('degraded')
# question text per scenario (the robust cross-notebook join key — gt-row-NN indices differ
# between notebooks because some filter the dataset, but the question text is stable).
_q_by_sid = {s['scenario_id']: (s['turns'][0].get('input', '') if s.get('turns') else '')
             for s in _specs}
# Ground-truth expected SQL + result, keyed by question text, so the evidence
# record pairs the agent's generated SQL/result with what was expected.
_gt_by_q = {row['Natural_Language_Question']: row for row in groundtruth}
per_scenario_agent = {}
for sid, d in _psa.items():
    q = _q_by_sid.get(sid, '')
    gt = _gt_by_q.get(q, {})
    gt_res = gt.get('Expected_SQL_Result')
    per_scenario_agent[sid] = {
        'question': q,
        'emitted_sql': d['sql_runs'] > 0,        # emitted SQL in at least one run
        'sql_run_fraction': round(d['sql_runs'] / len(d['tokens']), 4) if d['tokens'] else 0.0,
        'avg_total_tokens': round(sum(d['tokens']) / len(d['tokens'])) if d['tokens'] else 0,
        'avg_wall_clock_s': round(sum(d['latency']) / len(d['latency']), 2) if d['latency'] else None,
        # Evidence: agent's generated/executed SQL + actual result, vs ground truth.
        'generated_sql': d.get('gen_sql', ''),
        'executed_sql': d.get('exec_sql', ''),
        'degraded': d.get('degraded'),
        'result_row_count': d.get('rows'),
        'result_sample': d.get('sample', []),
        'expected_sql': gt.get('Expected_SQL_Query', ''),
        'expected_row_count': len(gt_res) if isinstance(gt_res, list) else None,
        'expected_sample': gt_res[:3] if isinstance(gt_res, list) else gt_res,
    }


print(f"=== k-run MEAN scores over EVAL_K={EVAL_K} run(s) ===")
display(df_mean)
print(f"\nMean agent cost/latency: avg_wall={agent_perf_mean['avg_wall_clock_s']}s  "
      f"total_tokens={agent_perf_mean['total_tokens']}")

# ── Persist the k-run mean summary (the file downstream notebooks 10 & 11 read). ──
kmean_file = f"../data/eval/results/ontology_query_kmean_eval_{timestamp}.json"
kmean = {
    'notebook': '6_ontology_query',
    'arm_label': 'vkg',                    # VKG path; for nb11 this is the WITH-OntologyRAG arm
    'agent': 'ontology_query_agent',
    'agent_id': agent_id,
    'runtime_arn': ONTOLOGY_QUERY_RUNTIME_ARN,
    'eval_id': EVAL_ID,
    'eval_k': EVAL_K,
    'evaluator_ids': ALL_EVALUATOR_IDS,
    'custom_evaluators': {'GoalSuccess': GOAL_SUCCESS_ID,
                          'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                          'SqlGrounded': SQL_GROUNDED_ID},
    'mean_scores': {row['Evaluator']: row['MeanScore'] for row in mean_rows},
    'std_scores': {row['Evaluator']: row['Std'] for row in mean_rows},
    'per_run_scores': [{'run': r['run'], 'batch_id': r['batch_id'],
                        'status': r['status'], 'scores': r['scores'],
                        'agent_perf': r['agent_perf']} for r in RUNS],
    'agent_perf_mean': agent_perf_mean,
    'per_scenario_goal_mean': per_scenario_goal_mean,
    'per_scenario_agent': per_scenario_agent,
}
kmean = _redact_account_ids(kmean)
with open(kmean_file, 'w') as f:
    json.dump(kmean, f, indent=2, default=str)
print(f"\n✓ Wrote k-run mean summary → {kmean_file}")

# ════════════════════════════════════════════════════════════════════════════
# B. Per-query DETAIL for the LAST run (batch_result + AGENT_RUNS point at run k).
#    Preserves the ontology_query_batch_eval_*.json file (downstream-compatible).
#
#    GUARD: batch_result is None when the final k-run was RESUMED from the resume
#    file (its live BatchEvaluation object isn't serialisable, so it can't survive
#    a kernel restart). In that case the k-mean headline above is already complete
#    and persisted; we simply skip the last-run drill-in rather than crash on a
#    None deref. Re-run cell 15 fresh (delete the resume file) to regenerate it.
# ════════════════════════════════════════════════════════════════════════════
if batch_result is None:
    print("\n⚠ Last-run detail SKIPPED — the final run was resumed from disk (no live "
          "batch result object). The k-run mean summary above is complete and saved. "
          "To regenerate the per-query last-run drill-in, delete the .resume_*.json "
          "file and re-run the batch cell for a fresh in-memory run.")
    combined_file = None
else:
    try:
        from agents.shared.eval_multiturn import group_runs_by_session
    except ImportError:
        import sys; sys.path.insert(0, '..')
        from agents.shared.eval_multiturn import group_runs_by_session

    print(f"\n── Last-run detail — Batch ID: {batch_result.batch_evaluation_id} "
          f"(status {batch_result.status}) ──")

    # ── 1. Aggregate per-evaluator scores (last run) ─────────────────────────
    agg_rows = []
    ev = batch_result.evaluation_results
    if ev is not None:
        print(f"Sessions: completed={ev.number_of_sessions_completed} "
              f"failed={ev.number_of_sessions_failed} total={ev.total_number_of_sessions}")
        for es in (ev.evaluator_summaries or []):
            stats = es.statistics
            avg = (f"{stats.average_score:.3f}"
                   if stats and stats.average_score is not None else None)
            agg_rows.append({'Evaluator': es.evaluator_id or 'unknown', 'AvgScore': avg,
                             'Evaluated': es.total_evaluated or 0, 'Failed': es.total_failed or 0})
    else:
        print("⚠ No aggregate evaluation_results — check job status / spans.")
    df_agg = pd.DataFrame(agg_rows)

    grouped = group_runs_by_session(AGENT_RUNS)

    # ── 2. Per-query evaluator events (last run) — reuse the helper from cell 15 ──
    event_rows = _event_rows_for_run(batch_result)
    def _f(e, key):
        return e[key] if key in e else (e.get('attributes', {}) or {}).get(key)
    _explan = {}
    for e in _fetch_events(batch_result):
        sess = _f(e, 'session.id') or _f(e, 'gen_ai.session.id')
        name = _f(e, 'gen_ai.evaluation.name')
        _explan[(_scenario_id_for_session(sess), _EVAL_NAME.get(name, name))] = \
            (str(_f(e, 'gen_ai.evaluation.explanation') or ''))
    for row in event_rows:
        row['explanation'] = _explan.get((row['scenario_id'], row['evaluator']), '')
    df_events = pd.DataFrame(event_rows)
    print(f"Per-query evaluator events (last run): {len(df_events)}")

    # ── 3. Per-turn trajectory table + clarification summary (last run) ──────
    turn_rows = []
    for sess, turns in grouped.items():
        sid = _scenario_id_for_session(sess)
        for t in turns:
            turn_rows.append({
                'scenario_id': sid, 'turn': t.get('turn_idx'),
                'message': (t.get('message') or ''), 'clarified': t.get('clarified'),
                'has_sql': bool(t.get('agent_sql')), 'wall_s': t.get('wall_clock_s'),
                'error': t.get('invoke_error'),
            })
    df_agent = pd.DataFrame(turn_rows)
    if not df_agent.empty:
        df_agent = df_agent.sort_values(['scenario_id', 'turn']).reset_index(drop=True)

    print("\n── Trajectory summary (last run, per scenario) ──")
    for sess, turns in grouped.items():
        sid = _scenario_id_for_session(sess)
        clar = [t.get('turn_idx') for t in turns if t.get('clarified')]
        final_sql = bool(turns[-1].get('agent_sql')) if turns else False
        print(f"  {sid}: {len(turns)} turn(s) · clarified_on={clar or 'none'} · "
              f"re_clarified={len(clar) > 1} · final_sql={final_sql}")

    # ── 4. Agent cost/latency (last run) ─────────────────────────────────────
    _lat = [r['wall_clock_s'] for r in AGENT_RUNS.values() if r.get('wall_clock_s') is not None]
    def _usage_sum(key):
        return sum(int((r.get('usage') or {}).get(key) or 0) for r in AGENT_RUNS.values())
    agent_perf = {
        'turns': len(AGENT_RUNS), 'sessions': len(grouped),
        'avg_wall_clock_s': round(sum(_lat) / len(_lat), 2) if _lat else None,
        'input_tokens': _usage_sum('inputTokens'), 'output_tokens': _usage_sum('outputTokens'),
        'total_tokens': _usage_sum('totalTokens'),
    }
    print(f"\n── Query agent cost/latency (last run) ──")
    print(f"  Avg wall-clock : {agent_perf['avg_wall_clock_s']}s  ·  tokens: {agent_perf['total_tokens']}")

    # ── Persist the last-run detail. Keep the ontology_query_batch_eval_ prefix so notebook 4
    #    (which globs query_batch_eval_*) does NOT pick up the VKG agent's results. ──
    combined_file = f"../data/eval/results/ontology_query_batch_eval_{timestamp}.json"
    combined = {
        'agent_id': agent_id, 'runtime_arn': ONTOLOGY_QUERY_RUNTIME_ARN, 'eval_id': EVAL_ID,
        'eval_k': EVAL_K, 'kmean_file': os.path.basename(kmean_file),
        'batch_evaluation_id': batch_result.batch_evaluation_id,
        'batch_evaluation_arn': batch_result.batch_evaluation_arn,
        'status': str(batch_result.status), 'evaluator_ids': ALL_EVALUATOR_IDS,
        'custom_evaluators': {'GoalSuccess': GOAL_SUCCESS_ID,
                              'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                              'SqlGrounded': SQL_GROUNDED_ID},
        'aggregate_summaries': agg_rows, 'per_query_events': event_rows,
        'agent_runs': list(AGENT_RUNS.values()), 'agent_performance': agent_perf,
    }
    combined = _redact_account_ids(combined)
    with open(combined_file, 'w') as f:
        json.dump(combined, f, indent=2, default=str)
    print(f"✓ Wrote last-run detail → {combined_file}")

    print("\n=== Aggregate per-evaluator scores (last run) ===")
    display(df_agg)
    print("=== Per-query evaluator scores (last run) ===")
    display(df_events)
    print("=== Per-turn trajectory (last run) ===")
    display(df_agent)


=== k-run MEAN scores over EVAL_K=3 run(s) ===


,Evaluator,MeanScore,Std,Runs,PerRun
0,GoalSuccess,0.71,0.0283,3,"[0.69, 0.75, 0.69]"
1,FinalAnswerFaithfulness,0.71,0.0283,3,"[0.69, 0.75, 0.69]"
2,SqlGrounded,1.00,0.0000,3,"[1.0, 1.0, 1.0]"



Mean agent cost/latency: avg_wall=40.9433s  total_tokens=1238375.6667

✓ Wrote k-run mean summary → ../data/eval/results/ontology_query_kmean_eval_20260701_114640.json

── Last-run detail — Batch ID: ontquery_gt_batch_r3_9ebcb654-04ad91e886 (status COMPLETED) ──
Sessions: completed=16 failed=0 total=16


Per-query evaluator events (last run): 48

── Trajectory summary (last run, per scenario) ──
  gt-row-02: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=True
  gt-row-00: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=True
  gt-row-01: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=True
  gt-row-03: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=True
  gt-row-04: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=False
  gt-row-05: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=True
  gt-row-07: 1 turn(s) · clarified_on=[0] · re_clarified=False · final_sql=False
  gt-row-06: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=True
  mt-parties-clarify: 2 turn(s) · clarified_on=[0] · re_clarified=False · final_sql=True
  gt-row-08: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=True
  mt-no-spurious-clarify: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql

,Evaluator,AvgScore,Evaluated,Failed
0,SqlGrounded_945bd536-PVILMn3oj9,1.000,16,0
1,GoalSuccess_945bd536-VjsoKqDtWq,0.690,16,0
2,FinalAnswerFaithfulness_945bd536-v5PmyP5Iqd,0.690,16,0


=== Per-query evaluator scores (last run) ===


,scenario_id,question,evaluator,score,label,explanation
0,gt-row-13,What are some common metrics that could be calculated with this data?,FinalAnswerFaithfulness,1.0,pass,"The final answer explains that no governed metrics are yet defined for this layer, then describes a wide range of computable metrics (counts, totals, distributions, allocations) across policies, parties, holdings/coverage, and distribution/fund data. It explicitly avoids returning any specific numeric result, matching the expected answer's requirement to describe computable metrics without numeric output. This aligns well with the expected answer's content and constraints."
1,gt-row-13,What are some common metrics that could be calculated with this data?,SqlGrounded,1.0,pass,"No executed SQL is present in the context; the agent provided a descriptive answer about potential metrics without running any query against Athena. Since there is no SQL execution to check against the ontology context, the grounding invariant is upheld by default (degraded/no-SQL case)."
2,gt-row-13,What are some common metrics that could be calculated with this data?,GoalSuccess,1.0,pass,"The final-answer record provides a descriptive list of computable metrics (coverage/policy counts and totals, financial totals, party counts, distribution metrics, investment allocations, underwriting distributions) without returning any specific numeric result, and explicitly notes no governed metrics are yet defined for this layer. This matches the expected answer: a description of computable metrics and governed metrics status, describing what is computable rather than a numeric result. No clarification was needed and none was inappropriately given. The trajectory satisfies the assertion."
3,gt-row-04,"For policies that have a payout schedule, show the policyholder's name, the payout frequency, and the projected next-payout amount.",FinalAnswerFaithfulness,0.0,fail,"The session context provided contains no actual conversation content—no user prompts, agent responses, or tool executions. There is no final-answer record span, and no natural-language assistant message to evaluate. Since the conversation never reaches a substantive answer (it's entirely empty), the score must be 0 regardless of the expected answer's content."
4,gt-row-04,"For policies that have a payout schedule, show the policyholder's name, the payout frequency, and the projected next-payout amount.",SqlGrounded,1.0,pass,"The session context is empty - there is no user prompt, no agent response, no tool execution history, no retrieved ontology context, and no executed SQL present. Since there is no executed SQL in the context at all, the agent did not run any ungrounded SQL against Athena. Per the degraded-run rule, absence of executed SQL means the grounding invariant was upheld by default (no SQL executed = degraded/empty run), so this passes as 'no SQL executed (degraded)'."
5,gt-row-04,"For policies that have a payout schedule, show the policyholder's name, the payout frequency, and the projected next-payout amount.",GoalSuccess,0.0,fail,"The session context provided is empty - there are no user prompts, agent responses, or tool execution history. There is no final-answer record to evaluate, and no conversation content to reconstruct. Since no evidence exists that the agent produced the expected answer about the 5 policies with payout schedules, the assertion cannot be verified as satisfied."
6,gt-row-14,What can I ask about this semantic layer?,FinalAnswerFaithfulness,1.0,pass,"The final answer describes the semantic layer's coverage areas (parties, policies, coverages/riders, financial activity, investments, distribution, reference data) and explicitly states there are no governed/certified metrics yet, only schema-computable concepts. It returns no specific data values. This matches the expected answer's description of question types and governed metrics availability (noting none are governed). The content aligns well with th

=== Per-turn trajectory (last run) ===


,scenario_id,turn,message,clarified,has_sql,wall_s,error
0,gt-row-00,0,Show me policies where the insured party is also the policyholder.,False,True,61.70,None
1,gt-row-01,0,"For each rider, who are the insured participants and what are their roles?",False,True,91.41,None
2,gt-row-02,0,List the top 5 most common party types and their human-readable descriptions.,False,True,36.23,None
3,gt-row-03,0,"What is the total market value of all active holdings, grouped by party?",False,True,91.43,None
4,gt-row-04,0,"For policies that have a payout schedule, show the policyholder's name, the payout frequency, and the projected next-payout amount.",False,False,102.54,None
5,gt-row-05,0,How many parties are there?,False,True,75.18,None
6,gt-row-06,0,List the top 10 coverage products by name.,False,True,54.95,None
7,gt-row-07,0,"Show the top 10 parties by total holding market value, including the investment product names they hold.",True,False,15.69,None
8,gt-row-08,0,What was the total financial-activity amount per month in 2024?,False,True,31.96,None
9,gt-row-13,0,What are some common metrics that could be calculated with this data?,False,False,21.17,None


## 8. Store IDs for Downstream Notebooks

Save the batch evaluation id, the per-query session ids, and the agent id so downstream
notebooks can reference this run.

In [11]:
ontology_query_batch_id = batch_result.batch_evaluation_id
ontology_query_eval_session_ids = list(AGENT_RUNS.keys())
ondemand_query_agent_id = agent_id           # kept name for downstream notebook compatibility
ondemand_query_session_id = (ontology_query_eval_session_ids[0]
                             if ontology_query_eval_session_ids else None)
%store ontology_query_batch_id
%store ontology_query_eval_session_ids
%store ondemand_query_agent_id
%store ondemand_query_session_id

print("✓ Stored for downstream notebooks:")
print(f"  ontology_query_batch_id         = {ontology_query_batch_id}")
print(f"  ontology_query_eval_session_ids = {len(ontology_query_eval_session_ids)} session(s)")
print(f"  ondemand_query_agent_id         = {ondemand_query_agent_id}")
print(f"\n  Combined results: {combined_file}")

Stored 'ontology_query_batch_id' (str)
Stored 'ontology_query_eval_session_ids' (list)
Stored 'ondemand_query_agent_id' (str)
Stored 'ondemand_query_session_id' (str)
✓ Stored for downstream notebooks:
  ontology_query_batch_id         = ontquery_gt_batch_r3_9ebcb654-04ad91e886
  ontology_query_eval_session_ids = 19 session(s)
  ondemand_query_agent_id         = semantic_layer_dev_ontology_query-HKGVpkE6XK

  Combined results: ../data/eval/results/ontology_query_batch_eval_20260701_114640.json


## Summary

This notebook evaluates the deployed **Ontology Query Agent** (VKG path) **entirely
server-side** via AgentCore Batch Evaluations. The only client-side work is invoking the
agent once per ground-truth row (required to produce spans) and reading its response for
cost/latency.

### Pipeline
1. **Custom evaluators** (LLM-as-Judge, no Lambda) — all **SESSION-level**:
   `GoalSuccess`, `FinalAnswerFaithfulness`, and `SqlGrounded`, created with
   `create_evaluator` and scored in-service.
2. **Dataset** — one `PredefinedScenario` per ground-truth row (per-query sessions);
   multi-turn rows carry several `Turn`s. The expected answer is threaded into `assertions`
   so the SESSION judges can read it; each scenario also carries `expected_trajectory`.
3. **agent_invoker** — drives the chat-stream path (reads + persists DDB history) so turn N+1
   resolves turn N's clarification; records the agent's own tokens / `runtimeMs` / wall-clock
   per turn into `AGENT_RUNS`.
4. **BatchEvaluationRunner** — one `StartBatchEvaluation` job over the SESSION-only evaluator
   set.
5. **Results** — aggregate per-evaluator scores, **per-query** scores via
   `fetch_evaluation_events`, and **per-query** agent cost/latency.

### Why SESSION-level evaluators (consistent with notebook 2)
This agent is multi-turn: it may answer in one turn, or ask a **clarifying question** first
and answer on a later turn. A SESSION-level judge reads the whole conversation via `{context}`
and scores the **final** answer once — the correct granularity for a multi-turn agent. So the
per-turn `Builtin.Correctness` / `Builtin.Helpfulness` builtins and the TRACE
`QueryAnswerFaithfulness` judge are **not** used (a TRACE judge errors on a clarification turn,
which makes no model/tool call, and fails the whole session); the per-turn trajectory table
(clarified / has_sql / latency) covers per-turn behaviour instead.

### What batch can / can't give you
- ✅ All three SESSION custom evaluator scores, server-side (no local scoring).
- ✅ **SQL grounding** scored server-side: the `{context}` placeholder exposes the session's
  phase outputs (the executed SQL and the retrieved ontology slice), so `SqlGrounded` reads
  them directly — no code-based/Lambda evaluator required.
- ✅ Per-query evaluator scores (via the output log stream) and per-query agent token/latency.
- ❌ The agent's own tokens are **not** in the batch result — captured by the invoker instead.

### Next steps
- Run the comparison notebooks (6–9) for cross-path / cross-source evaluation.